In [240]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

In [241]:
data = pd.read_csv("training_data_lowercase.csv",sep='\t', names=['label', 'title'])
print(data.shape)
data.fillna("",inplace=True)
print(data.head())


(34152, 2)
   label                                              title
0      0  donald trump sends out embarrassing new year‚s...
1      0  drunk bragging trump staffer started russian c...
2      0  sheriff david clarke becomes an internet joke ...
3      0  trump is so obsessed he even has obama‚s name ...
4      0  pope francis just called out donald trump duri...


as part of pre proc we are able to see color codes in csv which are incorrectly interpretted by vs code
they are not color code but simply corresponds to episode num

we also see video/picture are there in some data points, while these are just metadata it could change the meaning of the sentence once we remove punctuations

we also see that this metadata in enclosed in () in Training sample while its enclosed in [] in testing data, so we might need different pre processing

In [242]:
from preProc import normalize_text

data["clean_text"] = data["title"].apply(normalize_text)
print(data.head)

<bound method NDFrame.head of        label                                              title  \
0          0  donald trump sends out embarrassing new year‚s...   
1          0  drunk bragging trump staffer started russian c...   
2          0  sheriff david clarke becomes an internet joke ...   
3          0  trump is so obsessed he even has obama‚s name ...   
4          0  pope francis just called out donald trump duri...   
...      ...                                                ...   
34147      1  tears in rain as thais gather for late king's ...   
34148      1  pyongyang university needs non-u.s. teachers a...   
34149      1  philippine president duterte to visit japan ah...   
34150      1  japan's abe may have won election\tbut many do...   
34151      1  demoralized and divided: inside catalonia's po...   

                                              clean_text  
0      donald trump sends out embarrassing new year e...  
1      drunk bragging trump staffer started rus

In [243]:
from preProc import remove_stopwords


data["no_stopwords"] = data["clean_text"].apply(remove_stopwords)
print(data.head)



<bound method NDFrame.head of        label                                              title  \
0          0  donald trump sends out embarrassing new year‚s...   
1          0  drunk bragging trump staffer started russian c...   
2          0  sheriff david clarke becomes an internet joke ...   
3          0  trump is so obsessed he even has obama‚s name ...   
4          0  pope francis just called out donald trump duri...   
...      ...                                                ...   
34147      1  tears in rain as thais gather for late king's ...   
34148      1  pyongyang university needs non-u.s. teachers a...   
34149      1  philippine president duterte to visit japan ah...   
34150      1  japan's abe may have won election\tbut many do...   
34151      1  demoralized and divided: inside catalonia's po...   

                                              clean_text  \
0      donald trump sends out embarrassing new year e...   
1      drunk bragging trump staffer started r

In [244]:
from preProc import tokens_lemm, tokens_stemm

data["stemmed"] = data["no_stopwords"].apply(tokens_stemm)
data["lemmed"] = data["no_stopwords"].apply(tokens_lemm)

print(data.head)


<bound method NDFrame.head of        label                                              title  \
0          0  donald trump sends out embarrassing new year‚s...   
1          0  drunk bragging trump staffer started russian c...   
2          0  sheriff david clarke becomes an internet joke ...   
3          0  trump is so obsessed he even has obama‚s name ...   
4          0  pope francis just called out donald trump duri...   
...      ...                                                ...   
34147      1  tears in rain as thais gather for late king's ...   
34148      1  pyongyang university needs non-u.s. teachers a...   
34149      1  philippine president duterte to visit japan ah...   
34150      1  japan's abe may have won election\tbut many do...   
34151      1  demoralized and divided: inside catalonia's po...   

                                              clean_text  \
0      donald trump sends out embarrassing new year e...   
1      drunk bragging trump staffer started r

In [245]:
from sklearn.model_selection import train_test_split
X=data.drop(columns=['label'])
y=data['label']

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(f"Training set size: {X_train.shape}")
print(f"Testing set size: {X_val.shape}")
print(f"Training output size: {y_train.shape}")
print(f"Testing output size: {y_val.shape}")

Training set size: (27321, 5)
Testing set size: (6831, 5)
Training output size: (27321,)
Testing output size: (6831,)


BOW

In [246]:
from sklearn.feature_extraction.text import CountVectorizer
bow_vectorizer = CountVectorizer(max_features=5000)
X_train_BOW_L = bow_vectorizer.fit_transform(X_train["lemmed"])
X_val_BOW_L = bow_vectorizer.transform(X_val["lemmed"])
X_train_BOW_S = bow_vectorizer.fit_transform(X_train["stemmed"])
X_val_BOW_S = bow_vectorizer.transform(X_val["stemmed"])
X_train_BOW_C = bow_vectorizer.fit_transform(X_train["clean_text"])
X_val_BOW_C = bow_vectorizer.transform(X_val["clean_text"])

BIGRAM

In [247]:
bigram_vectorizer = CountVectorizer(ngram_range=(1,2))
X_train_BOW2_L = bigram_vectorizer.fit_transform(X_train["lemmed"])
X_val_BOW2_L = bigram_vectorizer.transform(X_val["lemmed"])
X_train_BOW2_S = bigram_vectorizer.fit_transform(X_train["stemmed"])
X_val_BOW2_S = bigram_vectorizer.transform(X_val["stemmed"])
X_train_BOW2_C = bigram_vectorizer.fit_transform(X_train["clean_text"])
X_val_BOW2_C = bigram_vectorizer.transform(X_val["clean_text"])

TFIDF

In [248]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_train_TF_L = tfidf_vectorizer.fit_transform(X_train["lemmed"])
X_val__TF_L = tfidf_vectorizer.transform(X_val["lemmed"])
X_train_TF_S = tfidf_vectorizer.fit_transform(X_train["stemmed"])
X_val__TF_S = tfidf_vectorizer.transform(X_val["stemmed"])
X_train_TF_C = tfidf_vectorizer.fit_transform(X_train["clean_text"])
X_val__TF_C = tfidf_vectorizer.transform(X_val["clean_text"])

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier


from sklearn.metrics import accuracy_score, classification_report, confusion_matrix



experiments = {
    "BoW_stemm": (X_train_BOW_S, X_val_BOW_S),
    "BoW_lemm": (X_train_BOW_L, X_val_BOW_L),
    "BoW_Clean": (X_train_BOW_C, X_val_BOW_C),
    "BIGRAM_stemm": (X_train_BOW2_S, X_val_BOW2_S),
    "BIGRAM_lemm": (X_train_BOW2_L, X_val_BOW2_L),
    "BIGRAM_Clean": (X_train_BOW2_C, X_val_BOW2_C),
    "TF-IDF_stemm": (X_train_TF_S, X_val__TF_S),
    "TF-IDF_lemm": (X_train_TF_L, X_val__TF_L),
    "TF-IDF_Clean": (X_train_TF_C, X_val__TF_C),
}

models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Linear SVC": LinearSVC(),
    "Random Forest": RandomForestClassifier(n_estimators=100),
}

results = []

for name, (Xtr, Xva) in experiments.items():
    for model_name, model in models.items():
        model.fit(Xtr, y_train)
        preds = model.predict(Xva)
        acc = accuracy_score(y_val, preds)
        results.append((name, model_name, acc))
        print(f"{name} : {model_name} : {acc:.4f}")
 

results_df = pd.DataFrame(results, columns=["Text Vectorization", "Model", "Validation Accuracy"]).sort_values(
    by="Validation Accuracy", ascending=False
)
print("\nValidation results:")
display(results_df)

best_model_name = results_df.iloc[0]["Model"]
print("Best model:", best_model_name)

BoW_stemm : Naive Bayes : 0.9245
BoW_stemm : Logistic Regression : 0.9340
BoW_stemm : Linear SVC : 0.9284
BoW_stemm : Random Forest : 0.9128
BoW_lemm : Naive Bayes : 0.9270
BoW_lemm : Logistic Regression : 0.9340
BoW_lemm : Linear SVC : 0.9270
BoW_lemm : Random Forest : 0.9129
BoW_Clean : Naive Bayes : 0.9409
BoW_Clean : Logistic Regression : 0.9477
BoW_Clean : Linear SVC : 0.9368
BoW_Clean : Random Forest : 0.9214
BIGRAM_stemm : Naive Bayes : 0.9356
BIGRAM_stemm : Logistic Regression : 0.9417
BIGRAM_stemm : Linear SVC : 0.9406
BIGRAM_stemm : Random Forest : 0.9250
BIGRAM_lemm : Naive Bayes : 0.9390
BIGRAM_lemm : Logistic Regression : 0.9433
BIGRAM_lemm : Linear SVC : 0.9423
BIGRAM_lemm : Random Forest : 0.9226
BIGRAM_Clean : Naive Bayes : 0.9492
BIGRAM_Clean : Logistic Regression : 0.9529
BIGRAM_Clean : Linear SVC : 0.9507
BIGRAM_Clean : Random Forest : 0.9354
TF-IDF_stemm : Naive Bayes : 0.9211
TF-IDF_stemm : Logistic Regression : 0.9315
TF-IDF_stemm : Linear SVC : 0.9344
TF-IDF_stem

,Text Vectorization,Model,Validation Accuracy
21,BIGRAM_Clean,Logistic Regression,0.952862
22,BIGRAM_Clean,Linear SVC,0.950666
20,BIGRAM_Clean,Naive Bayes,0.949202
9,BoW_Clean,Logistic Regression,0.947738
34,TF-IDF_Clean,Linear SVC,0.946128
17,BIGRAM_lemm,Logistic Regression,0.943347
33,TF-IDF_Clean,Logistic Regression,0.942761
18,BIGRAM_lemm,Linear SVC,0.942322
13,BIGRAM_stemm,Logistic Regression,0.941736
8,BoW_Clean,Naive Bayes,0.940858


Best model: Logistic Regression
